# YOLO-OBB Per-Detection Eigen-CAM Review Crops

Run this in Google Colab after `ParkingYoloOBB_review_crops.ipynb` has already created `review_crops/manifest.csv` and `review_crops/all/`.

This notebook creates one ROI-targeted CAM overlay per detected OBB in `review_crops_cam/`. Rebuild the website later from your local repo.
This also saves full-tile per-detection heatmaps to `review_crops_cam/tile_heatmaps/` and raw grayscale maps to `review_crops_cam/tile_heatmaps_raw/`.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 2. Install Dependencies

In [ ]:
!pip -q install ultralytics pillow numpy

## 3. Paths

These match the existing project layout in Google Drive.

In [ ]:
from pathlib import Path

BASE = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')
RUN = BASE / 'GeoreferenceTest/202606121032215360267'
SCRIPTS = BASE / 'scripts'

MODEL = RUN / 'runs/obb/train/weights/best.pt'
IMAGES = RUN / 'dataset/images/val'
MANIFEST = RUN / 'review_crops/manifest.csv'
CAM_OUTPUT = RUN / 'review_crops_cam'
TILE_HEATMAP_DIR = CAM_OUTPUT / 'tile_heatmaps'
TILE_HEATMAP_RAW_DIR = CAM_OUTPUT / 'tile_heatmaps_raw'

print('BASE:', BASE, BASE.exists())
print('RUN:', RUN, RUN.exists())
print('SCRIPTS:', SCRIPTS, SCRIPTS.exists())
print('MODEL:', MODEL, MODEL.exists())
print('IMAGES:', IMAGES, IMAGES.exists())
print('MANIFEST:', MANIFEST, MANIFEST.exists())
print('CAM_OUTPUT:', CAM_OUTPUT)
print('TILE_HEATMAP_DIR:', TILE_HEATMAP_DIR)
print('TILE_HEATMAP_RAW_DIR:', TILE_HEATMAP_RAW_DIR)


## 4. Sanity Check Inputs

In [ ]:
if not MODEL.exists():
    raise RuntimeError('Missing best.pt. Run YOLO-OBB training first in this Colab runtime.')
if not IMAGES.exists():
    raise RuntimeError(f'Missing validation images folder: {IMAGES}')
if not MANIFEST.exists():
    raise RuntimeError(f'Missing review crop manifest: {MANIFEST}')
if not (SCRIPTS / '13_generate_obb_cam_crops.py').exists():
    raise RuntimeError('Missing scripts/13_generate_obb_cam_crops.py in Google Drive. Upload the updated scripts folder first.')

print('validation images:', len(list(IMAGES.glob('*.png'))))
print('manifest rows:', max(0, len(MANIFEST.read_text().splitlines()) - 1))

## 5. Generate 20 Per-Detection CAM Overlays First

Run this first. It is the quick test.

In [ ]:
!python {SCRIPTS / '13_generate_obb_cam_crops.py'} \
  --model {MODEL} \
  --images {IMAGES} \
  --manifest {MANIFEST} \
  --output {CAM_OUTPUT} \
  --imgsz 1024 \
  --device 0 \
  --save-tile-heatmaps \
  --limit 20

## 6. Preview CAM Overlays

In [ ]:
from IPython.display import Image, display

cam_files = sorted((CAM_OUTPUT / 'all').glob('*.png'))
tile_cam_files = sorted((CAM_OUTPUT / 'tile_heatmaps').glob('*.png'))
raw_tile_cam_files = sorted((CAM_OUTPUT / 'tile_heatmaps_raw').glob('*.png'))
print('cropped CAM files:', len(cam_files))
print('full-tile CAM overlays:', len(tile_cam_files))
print('full-tile raw CAMs:', len(raw_tile_cam_files))

print('Cropped CAM preview:')
for path in cam_files[:12]:
    print(path.name)
    display(Image(filename=str(path)))

print('Full-tile CAM preview:')
for path in tile_cam_files[:3]:
    print(path.name)
    display(Image(filename=str(path)))


## 7. Generate All Per-Detection CAM Overlays

If the preview looks okay, run this full version.

In [ ]:
!python {SCRIPTS / '13_generate_obb_cam_crops.py'} \
  --model {MODEL} \
  --images {IMAGES} \
  --manifest {MANIFEST} \
  --output {CAM_OUTPUT} \
  --imgsz 1024 \
  --device 0 \
  --save-tile-heatmaps

## 8. Final Check

After this, download/sync `review_crops_cam/` to your local repo and rebuild the website locally.

In [ ]:
cam_files = sorted((CAM_OUTPUT / 'all').glob('*.png'))
tile_heatmap_files = sorted((CAM_OUTPUT / 'tile_heatmaps').glob('*.png'))
tile_heatmap_raw_files = sorted((CAM_OUTPUT / 'tile_heatmaps_raw').glob('*.png'))
manifest_cam = CAM_OUTPUT / 'manifest_cam.csv'

print('cropped CAM overlays:', len(cam_files))
print('full-tile CAM overlays:', len(tile_heatmap_files))
print('full-tile raw CAMs:', len(tile_heatmap_raw_files))
print('manifest_cam:', manifest_cam, manifest_cam.exists())
print('output folder:', CAM_OUTPUT)